# 8 - Linked views and layouts

Composition **across maps** lives in module-level functions:

- `panels` - lay maps out as one figure.
- `tabs` - one map per tab.
- `swipe` - a draggable before/after divider.
- `linked_views` - an overview map whose range tool drives a detail map (`RangeToolLink`).
- `grid` - a `GridSpace` of small multiples that pan together.

**Setup** and data - German population in 2010 vs 2020, and four months of global temperature.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # deliberately silence all warnings to keep the rendered docs clean

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import numpy as np
from digitalearth.interactive import panels, tabs, swipe, linked_views, grid

pop10 = Dataset.read_file(str(DATA / "germany_pop_2010_1km.tif"))
pop20 = Dataset.read_file(str(DATA / "germany_pop_2020_1km.tif"))
_v = pop20.read_array(band=0)
cap = float(np.nanpercentile(_v[_v > 0], 99))

def pop_map(ds, title):
    return InteractiveMap(crs=ds.epsg, title=title).field(ds, cmap="inferno", clim=(0.0, cap)).colorbar()

print("population rasters ready | p99:", round(cap))

## `panels` - side by side

`panels` composes maps into one `Layout`; `cols` wraps them into a grid.

In [ ]:
panels(pop_map(pop10, "2010"), pop_map(pop20, "2020"))

## `tabs` - one map per tab

`tabs(**named)` puts each named map on its own tab.

In [ ]:
tabs(**{"2010": pop_map(pop10, "2010"), "2020": pop_map(pop20, "2020")})

## `swipe` - a before/after divider

`swipe` reveals one map over another with a draggable divider - here 2010 under 2020.

In [ ]:
swipe(pop_map(pop10, "2010"), pop_map(pop20, "2020"))

## `linked_views` - an overview drives a detail

`linked_views(detail, overview)` wires a `RangeToolLink` so the overview's range box pans/zooms the detail
map.

In [ ]:
linked_views(pop_map(pop20, "detail"), pop_map(pop20, "overview"))

## `grid` - linked small multiples

`grid` lays maps out as a `GridSpace` that shares one set of axes across every cell.

In [ ]:
def temp_map(month, name):
    ds = Dataset.read_file(str(DATA / "global" / f"ncep_tavg_{month:02d}.tif"))
    return InteractiveMap(crs=ds.epsg, title=name).field(ds, cmap="RdBu_r")

gs = grid(temp_map(1, "Jan"), temp_map(4, "Apr"), temp_map(7, "Jul"), temp_map(10, "Oct"), cols=2)
print("grid cells:", sorted(gs.keys()))
gs